# Load the dataset

In [1]:
import pandas as pd

df = pd.read_csv("kyphosis.csv")
df.head()

,Kyphosis,Age,Number,Start
0,absent,71,3,5
1,absent,158,3,14
2,present,128,4,5
3,absent,2,5,1
4,absent,1,4,15


# Shape of the dataset

In [2]:
df.shape

(81, 4)

# Duplicate rows

In [4]:
print(df.duplicated().sum())

0


# Null values of the dataset

In [6]:
df.isnull().sum()

Kyphosis    0
Age         0
Number      0
Start       0
dtype: int64

# Data types

In [9]:
df.dtypes

Kyphosis      str
Age         int64
Number      int64
Start       int64
dtype: object

# Encode the Target Variable

In [11]:
df['Kyphosis'].value_counts()

Kyphosis
absent     64
present    17
Name: count, dtype: int64

In [12]:
df["Kyphosis"] = df["Kyphosis"].map({"absent": 0, "present": 1})
df.head()

,Kyphosis,Age,Number,Start
0,0,71,3,5
1,0,158,3,14
2,1,128,4,5
3,0,2,5,1
4,0,1,4,15


# Separate Features (X) and Target (y)

In [13]:
X = df.drop("Kyphosis", axis=1)
y = df["Kyphosis"]

# Split into Training and Testing Sets

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the data

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression Model

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train)

log_y_pred = log_model.predict(X_test_scaled)

print("Confusion Matrix\n")
print(confusion_matrix(y_test, log_y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, log_y_pred))

Confusion Matrix

[[13  0]
 [ 3  1]]

Classification Report

              precision    recall  f1-score   support

           0       0.81      1.00      0.90        13
           1       1.00      0.25      0.40         4

    accuracy                           0.82        17
   macro avg       0.91      0.62      0.65        17
weighted avg       0.86      0.82      0.78        17



# Decision Tree Classifier Model

In [27]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train_scaled, y_train)

tree_y_pred = tree_model.predict(X_test_scaled)

print("Confusion Matrix\n")
print(confusion_matrix(y_test, tree_y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, tree_y_pred))

Confusion Matrix

[[12  1]
 [ 3  1]]

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.92      0.86        13
           1       0.50      0.25      0.33         4

    accuracy                           0.76        17
   macro avg       0.65      0.59      0.60        17
weighted avg       0.73      0.76      0.73        17



# Random Forest Classifier Model

In [32]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(random_state=42)
forest_model.fit(X_train_scaled, y_train)

forest_y_pred = forest_model.predict(X_test_scaled)

print("Confusion Matrix\n")
print(confusion_matrix(y_test, forest_y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, forest_y_pred))

Confusion Matrix

[[13  0]
 [ 3  1]]

Classification Report

              precision    recall  f1-score   support

           0       0.81      1.00      0.90        13
           1       1.00      0.25      0.40         4

    accuracy                           0.82        17
   macro avg       0.91      0.62      0.65        17
weighted avg       0.86      0.82      0.78        17



# Support Vector Classifier Model

In [33]:
from sklearn.svm import SVC

svc_model = SVC(random_state=42)
svc_model.fit(X_train_scaled, y_train)

svc_y_pred = svc_model.predict(X_test_scaled)

print("Confusion Matrix\n")
print(confusion_matrix(y_test, svc_y_pred))
print("\nClassification Report\n")
print(classification_report(y_test, svc_y_pred))

Confusion Matrix

[[13  0]
 [ 2  2]]

Classification Report

              precision    recall  f1-score   support

           0       0.87      1.00      0.93        13
           1       1.00      0.50      0.67         4

    accuracy                           0.88        17
   macro avg       0.93      0.75      0.80        17
weighted avg       0.90      0.88      0.87        17



# Hyperparameter tuning for the SVC model

In [36]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 1, 10, 100], 
    "kernel": ["linear", "rbf", "poly"],
    "gamma": ["scale", "auto", 0.1, 0.01],
    "class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(estimator=svc_model, param_grid=param_grid, cv=5, scoring='f1', verbose=1, n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print("\nBest Hyperparameters found by GridSearchCV:\n")
print(grid_search.best_params_)

best_svc_model = grid_search.best_estimator_
best_svc_y_pred = best_svc_model.predict(X_test_scaled)

print("\nTuned SVM Confusion Matrix:\n")
print(confusion_matrix(y_test, best_svc_y_pred))
print("\nTuned SVM Classification Report:\n")
print(classification_report(y_test, best_svc_y_pred))

Fitting 5 folds for each of 96 candidates, totalling 480 fits

Best Hyperparameters found by GridSearchCV:

{'C': 10, 'class_weight': 'balanced', 'gamma': 0.01, 'kernel': 'rbf'}

Tuned SVM Confusion Matrix:

[[10  3]
 [ 1  3]]

Tuned SVM Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.77      0.83        13
           1       0.50      0.75      0.60         4

    accuracy                           0.76        17
   macro avg       0.70      0.76      0.72        17
weighted avg       0.81      0.76      0.78        17



In [37]:
import joblib

joblib.dump(best_svc_model, 'kyphosis_svm_model.pkl')
joblib.dump(scaler, 'kyphosis_scaler.pkl')

['kyphosis_scaler.pkl']